In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- Load the Dataset ----
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_reviews = pd.read_csv("Processed_Reviews.csv")
print("Dataset shape:", df_reviews.shape)
print("\nFirst few rows:")
df_reviews.head()

# Check column names
print("\nColumn names:", df_reviews.columns.tolist())

# Check for null values
print("\nNull values:\n", df_reviews.isnull().sum())

# Based on the reviews in the file:
labels = [
    1,  # "The product arrived on time. Packaging was great, and the quality is amazing!" -> Positive
    1,  # "THIS PRODUCT IS JUST AMAZING! I LOVE IT." -> Positive
    1,  # "I bought this phone for $799, and it has a 120Hz display. Totally worth it!" -> Positive
    0,  # "Wow!!! This product is awesome... but a bit expensive??" -> Negative (expensive)
    1,  # "The laptop works perfectly fine." -> Positive
    0,  # "Check out the full product details here: ..." -> Neutral/Negative (just a link)
    1,  # "Great Purchase! I am happy with this product." -> Positive
    0,  # "The battery life is excellent, but the charging cable is too short." -> Negative (short cable)
    1,  # "I can't believe it's so good! Didn't expect such quality." -> Positive
    1,  # "Love this product! Fast delivery, amazing quality!" -> Positive
    1,  # "TBH, I wasn't expecting much, but OMG, this is awesome!!" -> Positive
    1,  # "This is the best product I have ever used in my life!" -> Positive
    1,  # "The shoes were comfortable, fitting nicely, and worked perfectly for jogging." -> Positive
]

# ---- PreProcessing ----
# Add the label column
df_reviews['label'] = labels
df_reviews['label_num'] = df_reviews['label'].map({1: 'positive', 0: 'negative'})

print("\nLabel distribution:")
print(df_reviews['label'].value_counts())
print(f"\nPositive reviews: {sum(df_reviews['label']==1)}")
print(f"Negative reviews: {sum(df_reviews['label']==0)}")

# Check the lemmatized column
print("\nLemmatized column sample:")
print(df_reviews['lemmatized'].head())

# Use only the lemmatized column for text classification
text_data = df_reviews['lemmatized']
labels_data = df_reviews['label']

print(f"\nText data shape: {len(text_data)}")
print(f"Labels shape: {len(labels_data)}")

from sklearn.feature_extraction.text import TfidfVectorizer

# ---- Vectorization ----

# Create TF-IDF Vectorizer
tfidf_vec = TfidfVectorizer(
    max_features=1000,  # Limit to top 1000 features
    min_df=2,           # Ignore terms that appear in less than 2 documents
    max_df=0.95,        # Ignore terms that appear in more than 95% of documents
    ngram_range=(1, 2)  # Use unigrams and bigrams
)

# Transform the text data into numerical features
X = tfidf_vec.fit_transform(text_data)
y = labels_data

print(f"\nTF-IDF matrix shape: {X.shape}")
print(f"Feature names sample: {list(tfidf_vec.get_feature_names_out())[:10]}")

# ---- Model Training -----
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

# Split data into training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Naive Bayes
nb_clf = MultinomialNB().fit(X_train, y_train)

# SVM
svm_clf = SVC(kernel='linear').fit(X_train, y_train)

# ---- Evaluation ----
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Make predictions
nb_predicted = nb_clf.predict(X_test)
svm_predicted = svm_clf.predict(X_test)

# Function to display evaluation metrics
def evaluate_model(y_true, y_pred, model_name):
    print(f"\n{'='*50}")
    print(f"{model_name} - Performance Evaluation")
    print(f"{'='*50}")
    
    # Accuracy
    accuracy = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Error Rate: {1-accuracy:.4f}")
    
    # Precision, Recall, F1-Score
    precision = precision_score(y_true, y_pred, average='binary')
    recall = recall_score(y_true, y_pred, average='binary')
    f1 = f1_score(y_true, y_pred, average='binary')
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    
    # Classification Report
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['Negative', 'Positive']))
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    print(f"Confusion Matrix:")
    print(cm)
    
    # Visualize Confusion Matrix
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.colorbar()
    tick_marks = [0, 1]
    plt.xticks(tick_marks, ['Negative', 'Positive'])
    plt.yticks(tick_marks, ['Negative', 'Positive'])
    
    # Add text annotations
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), 
                    horizontalalignment="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

# Evaluate Naive Bayes
nb_metrics = evaluate_model(y_test, nb_predicted, "Naive Bayes")

# Evaluate SVM
svm_metrics = evaluate_model(y_test, svm_predicted, "SVM")


Dataset shape: (13, 14)

First few rows:

Column names: ['Review', 'lowercased', 'urls_removed', 'html_removed', 'emojis_removed', 'slangs_replaced', 'contractions_replaced', 'punctuations_removed', 'numbers_removed', 'spelling_corrected', 'stopwords_removed', 'stemmed_words', 'lemmatized', 'tokenized']

Null values:
 Review                   0
lowercased               0
urls_removed             0
html_removed             0
emojis_removed           0
slangs_replaced          0
contractions_replaced    0
punctuations_removed     0
numbers_removed          0
spelling_corrected       0
stopwords_removed        0
stemmed_words            0
lemmatized               0
tokenized                0
dtype: int64

Label distribution:
label
1    10
0     3
Name: count, dtype: int64

Positive reviews: 10
Negative reviews: 3

Lemmatized column sample:
0    product arrive time packaging great quality am...
1                                   product amaze love
2                   buy phone hz display 